In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import NearestNeighbors
import re

In [2]:
df = pd.read_csv('dataset.csv')

In [3]:
def normalize_string(s):
    return re.sub(r'\W+', ' ', s).strip().lower()

def process_track_name(name):
    return normalize_string(re.sub(r'[-\(].*?[\)]', '', name))

def process_artist(artist):
    return ';'.join(sorted([normalize_string(a) for a in artist.split(';')]))

df['clean_track'] = df['track_name'].apply(process_track_name)
df['clean_artist'] = df['artists'].apply(process_artist)

features = [
    'popularity', 'danceability', 'energy', 'loudness',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo'
]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[features])
print(scaled_features)


[[ 1.97359932  0.65669378 -0.67284939 ...  0.69741576  0.95573562
  -1.13604934]
 [ 1.04759707 -0.78356805 -1.81343427 ... -0.598793   -0.74521932
  -1.48216305]
 [ 1.15048621 -0.68229964 -1.06722111 ... -0.51809517 -1.30334516
  -1.52056481]
 ...
 [-0.65007373  0.39227071 -1.1832128  ... -0.68503879  1.06204531
   0.339647  ]
 [ 0.3273731   0.15597776 -0.49886186 ...  0.25357774 -0.19089025
   0.45853646]
 [-0.65007373 -0.18720963 -0.57232326 ... -0.65780328  0.9291582
  -1.42543996]]


### KNN with all numerical features

In [4]:
knn = NearestNeighbors(n_neighbors=50, metric='cosine')
knn.fit(scaled_features)

def find_similar_songs(song_name, artist_name, n_results=5):
    clean_input_track = process_track_name(song_name)
    clean_input_artist = process_artist(artist_name)
    
    target_mask = (df['clean_track'] == clean_input_track) & \
                 (df['clean_artist'] == clean_input_artist)
    
    if not target_mask.any():
        print(f"Song '{song_name}' by {artist_name} not found in database")
        return
    
    target_idx = df[target_mask].index[0]
    target_genre = df.loc[target_idx, 'track_genre']
    
    distances, indices = knn.kneighbors([scaled_features[target_idx]], n_neighbors=50)
    
    results = []
    for i, idx in enumerate(indices[0]):
        if idx == target_idx:
            continue 
        
        same_artist = df.loc[idx, 'clean_artist'] == clean_input_artist
        same_base_track = process_track_name(df.loc[idx, 'track_name']) == clean_input_track
        if same_artist and same_base_track:
            continue
        
        results.append({
            'track': df.loc[idx, 'track_name'],
            'artist': df.loc[idx, 'artists'],
            'genre': df.loc[idx, 'track_genre'],
            'similarity': 1 - distances[0][i]  
        })
        
        if len(results) >= n_results:
            break
    
    print(f"\nTop {n_results} similar songs to '{song_name}' by {artist_name} ({target_genre}):")
    for i, item in enumerate(results, 1):
        print(f"{i}. {item['track']} by {item['artist']}")
        print(f"   Genre: {item['genre']} | Similarity: {item['similarity']:.3f}")
    
    return

find_similar_songs("Comedy", "Gen Hoshino")
find_similar_songs("You Belong With Me", "Taylor Swift")



Top 5 similar songs to 'Comedy' by Gen Hoshino (acoustic):
1. Blink 182 by Trille
   Genre: german | Similarity: 0.911
2. Elevated by Shubh
   Genre: hip-hop | Similarity: 0.910
3. Knee Socks by Arctic Monkeys
   Genre: garage | Similarity: 0.910
4. Fairytale by Livingston
   Genre: sad | Similarity: 0.909
5. 馬と鹿 by Kenshi Yonezu
   Genre: anime | Similarity: 0.907

Top 5 similar songs to 'You Belong With Me' by Taylor Swift (pop):
1. He's All I Need by Joe Smooth
   Genre: chicago-house | Similarity: 0.986
2. A quien le bailo yo by Baymont Bross;Estela Amaya
   Genre: breakbeat | Similarity: 0.985
3. America by Cooltime Kids
   Genre: kids | Similarity: 0.981
4. Hold My Hand by Jess Glynne
   Genre: house | Similarity: 0.980
5. In Your Arms (For An Angel) by Topic;Robin Schulz;Nico Santos;Paul van Dyk
   Genre: edm | Similarity: 0.980


### KNN with all numerical features with same genre 

In [5]:
knn = NearestNeighbors(n_neighbors=400, metric='cosine')
knn.fit(scaled_features)

def similar_songs_genre(song_name, artist_name, n_results=5):
    target_mask = (df['track_name'] == song_name) & (df['artists'] == artist_name)
    if not target_mask.any():
        print(f"Error: '{song_name}' by {artist_name} not found.")
        return []
    
    target_idx = df[target_mask].index[0]
    target_genre = df.loc[target_idx, 'genre_code']
    
    distances, indices = knn.kneighbors([scaled_features[target_idx]], n_neighbors=100)
    
    results = []
    for i, idx in enumerate(indices[0]):
        if idx == target_idx:
            continue
        
        current_genre = df.loc[idx, 'genre_code']
        same_artist = (df.loc[idx, 'artists'] == artist_name)
        similar_title = (
            song_name.lower() in df.loc[idx, 'track_name'].lower() or
            df.loc[idx, 'track_name'].lower() in song_name.lower()
        )
        
        if current_genre != target_genre:
            continue
        if same_artist and similar_title:
            continue
        
        results.append({
            'track': df.loc[idx, 'track_name'],
            'artist': df.loc[idx, 'artists'],
            'genre': current_genre,
            'similarity': 1 - distances[0][i]  
        })
        
        if len(results) >= n_results:
            break
    
    if len(results) < n_results:
        print(f"Warning: Only found {len(results)}/{n_results} songs in genre '{target_genre}'")
    
    print(f"\nTop {len(results)} similar songs to '{song_name}' ({target_genre}):")
    for i, item in enumerate(results, 1):
        print(f"{i}. {item['track']} by {item['artist']} | Similarity: {item['similarity']:.3f}")
    
    return

similar_songs_genre("Comedy", "Gen Hoshino")
similar_songs_genre("I'm Yours", "Jason Mraz")


Top 1 similar songs to 'Comedy' (0):
1. Days I Will Remember by Tyrone Wells | Similarity: 0.889

Top 2 similar songs to 'I'm Yours' (0):
1. Lucky by Jason Mraz;Colbie Caillat | Similarity: 0.929
2. Stand Your Ground by Joshua Hyslop | Similarity: 0.903
